In [5]:
import json
import numpy as np

r = 0.081      
L = 0.074       

log_file = "practical3.json"

cmd_data = []
true_data = []

with open(log_file, "r") as file:

    for line in file:

        try:
            data = json.loads(line)

            topic = data["topic_name"]
            timestamp = float(data["timestamp"])

            if topic == "/wheel_speeds_cmd":

                left = data["message"]["vector"]["x"]
                right = data["message"]["vector"]["y"]

                cmd_data.append([timestamp, left, right])

            elif topic == "/true_wheel_speeds":

                left = data["message"]["vector"]["x"]
                right = data["message"]["vector"]["y"]

                true_data.append([timestamp, left, right])

        except (json.JSONDecodeError, KeyError, TypeError, ValueError):
            continue


cmd_data = np.array(cmd_data)
true_data = np.array(true_data)

# match command speeds with nearest true speeds

linear_errors = []
angular_errors = []
time_differences = []

for cmd_timestamp, cmd_left, cmd_right in cmd_data:

    # Find the true wheel-speed sample closest in time
    index = np.argmin(
        np.abs(true_data[:, 0] - cmd_timestamp)
    )

    true_timestamp, true_left, true_right = true_data[index]

    dt = abs(true_timestamp - cmd_timestamp)

    # convert wheel speeds into linear velocity

    v_model = (r / 2) * (cmd_right + cmd_left)

    v_true = (r / 2) * (true_right + true_left)

    # convert wheel speeds into angular velocity

    w_model = (r / L) * (cmd_right - cmd_left)

    w_true = (r / L) * (true_right - true_left)

    # calculate motion model errors
    linear_error = v_model - v_true
    angular_error = w_model - w_true

    linear_errors.append(linear_error)
    angular_errors.append(angular_error)

    time_differences.append(dt)


linear_errors = np.array(linear_errors)
angular_errors = np.array(angular_errors)
time_differences = np.array(time_differences)

# Calculate variance of the errors - our proposed Q values
Q_DOTX_0 = np.var(linear_errors, ddof=1)

Q_DOTG_0 = np.var(angular_errors, ddof=1)

print("Linear velocity")

print("Mean error:",
      np.mean(linear_errors),
      "m/s")

print("Standard deviation:",
      np.std(linear_errors, ddof=1),
      "m/s")

print("Q_DOTX:",
      Q_DOTX_0) 
# in (m/s)^2


print("Angular velocity")

print("Mean error:",
      np.mean(angular_errors),
      "rad/s")

print("Standard deviation:",
      np.std(angular_errors, ddof=1),
      "rad/s")

print("Q_DOTG:",
      Q_DOTG_0)
# in (rad/s)^2

Linear velocity
Mean error: 0.0026308753459861074 m/s
Standard deviation: 0.037954812776439265 m/s
Q_DOTX: 0.0014405678128945575
Angular velocity
Mean error: 0.02420664522380992 rad/s
Standard deviation: 0.5704660644195659 rad/s
Q_DOTG: 0.3254315306543484
